In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline 
import pandas as pd

import scipy.signal

from astropy.io import ascii
import exoplanet as xo

import pymc3 as pm
import pymc3_ext as pmx
import lightkurve as lk
from astropy.timeseries import BoxLeastSquares, LombScargle
from astropy.stats import sigma_clip

In [ ]:
#Curva = ascii.read('Data_Transit_TIC427393298_Full.dat')

#Times = Curva.columns[0].astype(np.float64)
#Flux = Curva.columns[1].astype(np.float64)
curva = pd.read_csv("/home/yceb2024/Lucia/starry/pleyades_2/TIC_26017840/Tic 26017840_s_70_corrected.csv")
Times = curva["time"]
Flux = curva["flux"]

fig = plt.figure(figsize=(12,5), dpi=300)
plt.plot(Times, Flux, ".k")
plt.ylabel("Flujo")
plt.xlabel("Tiempo")
plt.rc('font', size=16)

## De-trending usando wotan

In [ ]:
from wotan import flatten
flatten_lc, trend_lc = flatten(Times, Flux, window_length=0.7, break_tolerance=2.0, method='hspline', return_trend=True)
#, method='biweight'
#window lenght es el porcentaje de puntos que se usan para hacer la ventana de observacion
fig = plt.figure(figsize=(12,5), dpi=300)
plt.plot(Times, Flux, ".k")
plt.plot(Times, trend_lc, c='b')
plt.ylabel("Flujo")
plt.xlabel("Tiempo")
plt.rc('font', size=12)

In [ ]:
#guardamos el detrending en un csv para el analisis posterior
data_trend = pd.DataFrame({"Time": Times, "Flux": trend_lc})
data_trend.to_csv("tic_26017840_trend_wotan.csv", index = False)

In [ ]:
Fl_Det = Flux/trend_lc

fig = plt.figure(figsize=(12,5), dpi=300)
#plt.plot(Time, Flux, ".k")
#plt.plot(Times, Fl_Det, c='b')
plt.scatter(Times, Fl_Det, c='b',s = 10)#s tamaño de punto 
plt.ylabel("Flujo")
plt.xlabel("Tiempo")
plt.rc('font', size=12)

# Sistema triple??

In [ ]:
# Quitar outliers fuertes (muy importante para BLS)
mask = np.isfinite(Fl_Det)
mask &= (Fl_Det > 0.9) & (Fl_Det < 1.1)

t = Times[mask]
f = Fl_Det[mask]

# Normalizar a media 0 para Lomb-Scargle
f_ls = f - np.median(f)


In [ ]:
from astropy.timeseries import LombScargle

freq, power = LombScargle(t, f_ls).autopower(
    minimum_frequency=0.05,   # periodos hasta 20 d
    maximum_frequency=10      # periodos hasta 0.1 d
)

period = 1/freq

plt.figure(figsize=(10,5), dpi=200)
plt.plot(period, power, 'k')
plt.xscale('log')
plt.xlabel("Periodo [días]")
plt.ylabel("Potencia LS")
plt.title("Lomb–Scargle Periodogram")
plt.gca().invert_xaxis()
plt.show()


In [ ]:
freq, power = LombScargle(t, f_ls).autopower(
    minimum_frequency=0.05,   # periodos hasta 20 d
    maximum_frequency=10      # periodos hasta 0.1 d
)

period = 1/freq

plt.figure(figsize=(10,5), dpi=200)
plt.plot(period, power, 'k')
plt.xlabel("Periodo [días]")
plt.ylabel("Potencia LS")
plt.title("Lomb–Scargle Periodogram")

plt.xlim(0.1, 20)   # opcional: enfocar el rango útil
plt.gca().invert_xaxis()  # opcional, para que periodos largos queden a la derecha

plt.show()

In [ ]:

model = BoxLeastSquares(t, f)

periods   = np.linspace(0.3, 10, 20000)
durations = np.linspace(0.02, 0.2, 15)

bls_power = model.power(periods, durations)

plt.figure(figsize=(10,5), dpi=100)
plt.plot(bls_power.period, bls_power.power, 'k')
plt.xlabel("Periodo [días]")
plt.ylabel("Potencia BLS")
plt.title("BLS Periodogram (Transit Search)")
plt.show()


In [ ]:
P = 2.77
t0 = t[np.argmin(f)]  # aproximación rápida

phase = ((t - t0 + 0.5*P) % P)/P - 0.5

plt.figure(figsize=(8,5), dpi=200)
plt.scatter(phase, f, s=5, color='k')
plt.xlabel("Fase")
plt.ylabel("Flujo")
plt.title("Curva plegada P = 2.77 d")
plt.show()


## Vamos de nuevo con otro analisis desde el principio

In [ ]:
#graficar la curva con los datos corregidos
curva = pd.read_csv("/home/yceb2024/Lucia/starry/pleyades_2/TIC_26017840/Tic 26017840_s_70_corrected.csv")
Times = curva["time"]
Flux = curva["flux"]

fig = plt.figure(figsize=(12,5), dpi=300)
plt.plot(Times, Flux, ".k")
plt.ylabel("Flujo")
plt.xlabel("Tiempo")
plt.rc('font', size=16)

In [ ]:
#Paso 1: limpieza
mask = np.isfinite(Times) & np.isfinite(Flux)
Times = Times[mask]
Flux  = Flux[mask]

# Quitar outliers extremos (rayos cósmicos, puntos malos)
med = np.median(Flux)
std = np.std(Flux)

mask = np.abs(Flux - med) < 5*std
Times = Times[mask]
Flux  = Flux[mask]

plt.figure(figsize=(12,4))
plt.plot(Times, Flux, ".k", ms=2)
plt.title("Curva limpia")
plt.xlabel("Tiempo")
plt.ylabel("Flujo")
plt.show()


In [ ]:
#paso 2: el detrending
from wotan import flatten

flatten_lc, trend_lc = flatten(
    Times, Flux,
    window_length=0.7,
    break_tolerance=2.0,
    method='hspline',
    return_trend=True
)

Fl_det = Flux / trend_lc

plt.figure(figsize=(12,4))
plt.plot(Times, Flux, '.k', alpha=0.4, label="Original")
plt.plot(Times, trend_lc, 'r', label="Tendencia")
plt.legend()
plt.title("Detrending")
plt.show()

plt.figure(figsize=(12,4))
plt.plot(Times, Fl_det, '.k', ms=2)
plt.title("Curva con detrending")
plt.xlabel("Tiempo")
plt.ylabel("Flujo normalizado")
plt.show()


In [ ]:
#paso3: rotación estelar (Lomb-Scargle)
from astropy.timeseries import LombScargle

f_ls = Fl_det - np.median(Fl_det)

freq, power = LombScargle(Times, f_ls).autopower(
    minimum_frequency=0.05,
    maximum_frequency=5
)

period_ls = 1/freq
best_period_ls = period_ls[np.argmax(power)]

print("Periodo de rotación estimado:", best_period_ls)

plt.figure(figsize=(10,4))
plt.plot(period_ls, power, 'k')
plt.axvline(best_period_ls, color='r', ls='--')
plt.xlabel("Periodo [días]")
plt.ylabel("Potencia")
plt.title("Lomb–Scargle")
plt.show()

In [ ]:
#Paso4: Búsqueda de tránsitos (BLS)
from astropy.timeseries import BoxLeastSquares

model = BoxLeastSquares(Times, Fl_det)

periods   = np.linspace(0.3, 10, 20000)
durations = np.linspace(0.02, 0.2, 15)

bls = model.power(periods, durations)

best_period_1 = bls.period[np.argmax(bls.power)]
print("Mejor periodo BLS:", best_period_1)

plt.figure(figsize=(10,4))
plt.plot(bls.period, bls.power, 'k')
plt.axvline(best_period_1, color='r', ls='--')
plt.xlabel("Periodo [días]")
plt.ylabel("Potencia BLS")
plt.title("BLS Periodogram")
plt.show()

In [ ]:
#Paso 5: Curva plegada
t0 = bls.transit_time[np.argmax(bls.power)]

phase = ((Times - t0 + 0.5*best_period_1) % best_period_1)/best_period_1 - 0.5

plt.figure(figsize=(8,5))
plt.scatter(phase, Fl_det, s=5, color='k')
plt.xlabel("Fase")
plt.ylabel("Flujo")
plt.title(f"Curva plegada P = {best_period_1:.4f} d")
plt.show()


In [ ]:
#paso6: Buscar segundo cuerpo (residuales)
model_curve = model.model(Times, best_period_1,
                          bls.duration[np.argmax(bls.power)],
                          t0)

residuals = Fl_det - model_curve + 1

model2 = BoxLeastSquares(Times, residuals)
bls2 = model2.power(periods, durations)

best_period_2 = bls2.period[np.argmax(bls2.power)]
print("Segundo periodo encontrado:", best_period_2)


In [ ]:
# Fuerza a usar SOLO estas versiones
t = np.array(Times)
flux = np.array(Flux)
trend = np.array(trend_lc)

print("Sizes originales:", len(t), len(flux), len(trend))

# Recalcular detrended flux desde cero
f_det = flux / trend

print("Size f_det nuevo:", len(f_det))

# Máscara consistente
mask = np.isfinite(t) & np.isfinite(f_det)

print("Size mask:", len(mask))

t_clean = t[mask]
f_clean = f_det[mask]

print("Sizes limpios:", len(t_clean), len(f_clean))

In [ ]:
P_real = 2 * 1.3854842742137106

phase = (t_clean % P_real) / P_real
order = np.argsort(phase)

plt.figure(figsize=(8,5), dpi=200)
plt.scatter(phase[order], f_clean[order], s=6, color='k')
plt.xlabel("Fase orbital")
plt.ylabel("Flujo normalizado")
plt.title("Curva plegada con P = 2.77 días")
plt.show()

In [ ]:
phase = (t_clean % P_real) / P_real
# Ventanas alrededor de cada eclipse
prim = (phase > 0.15) & (phase < 0.25)
sec  = (phase > 0.65) & (phase < 0.75)

depth_prim = 1 - np.median(f_clean[prim])
depth_sec  = 1 - np.median(f_clean[sec])

print("Profundidad primaria:", depth_prim)
print("Profundidad secundaria:", depth_sec)

In [ ]:
P_planet = 1.3854842742137106  # el de BLS
phase = (t_clean % P_planet) / P_planet
transit_mask = (phase > 0.65) & (phase < 0.75)
t_out = t_clean[~transit_mask]
f_out = f_clean[~transit_mask]


In [ ]:
freq, power = LombScargle(t_out, f_out).autopower(
    minimum_frequency=0.05,
    maximum_frequency=10
)

periods = 1 / freq

best_idx = np.argmax(power)
best_period = periods[best_idx]
best_power = power[best_idx]

print("Mejor periodo LS:", best_period, "días")
print("Potencia máxima:", best_power)

idx_sorted = np.argsort(power)[::-1]  # de mayor a menor

print("Top 5 periodos LS:")
for i in range(5):
    print(f"{i+1}: Periodo = {periods[idx_sorted[i]]:.5f} d  |  Power = {power[idx_sorted[i]]:.5f}")


plt.figure(figsize=(8,5), dpi=200)
plt.plot(periods, power, 'k')
plt.xlabel("Periodo [días]")
plt.ylabel("Potencia LS")
plt.title("Periodograma LS (sin tránsito principal)")
plt.gca().invert_xaxis()
plt.show()

In [ ]:
phase_res = (t_out % P_planet) / P_planet
order = np.argsort(phase_res)

plt.figure(figsize=(8,5), dpi=200)
plt.scatter(phase_res[order], f_out[order], s=6, color='k')
plt.xlabel("Fase")
plt.ylabel("Flujo (sin tránsito principal)")
plt.title("Curva plegada residual")
plt.show()

In [ ]:
# tiempos centrales estimados de tránsito
t0s = np.arange(t0, t0 + 20*P_planet, P_planet)

for tc in t0s:
    mask = abs(t_clean - tc) < 0.2
    plt.figure()
    plt.scatter(t_clean[mask] - tc, f_clean[mask], s=5)
    plt.title(f"Tránsito cerca de {tc:.2f}")
    plt.show()


In [ ]:
P = 1.3854842742137106
t0 = t_clean[np.argmin(f_clean)]

# Número de tránsito aproximado para cada punto
n_all = np.floor((t_clean - t0) / P + 0.5).astype(int)

unique_n = np.unique(n_all)
print("Número de tránsitos detectados:", len(unique_n))

In [ ]:
t_obs = []
n_list = []

for n in unique_n:
    mask = (n_all == n)
    
    # seleccionar solo puntos realmente en tránsito
    in_tr = mask & (f_clean < 1 - 0.002)   # ajusta umbral si hace falta
    
    if np.sum(in_tr) > 5:
        weights = 1 - f_clean[in_tr]
        t_center = np.sum(t_clean[in_tr] * weights) / np.sum(weights)
        
        t_obs.append(t_center)
        n_list.append(n)

t_obs = np.array(t_obs)
n_list = np.array(n_list)

print("Tránsitos medidos:", len(t_obs))

In [ ]:
t_calc = t0 + n_list * P
O_minus_C = t_obs - t_calc

In [ ]:
plt.figure(figsize=(8,4))
plt.scatter(n_list, O_minus_C * 24 * 60, color='k')
plt.axhline(0, color='r')
plt.xlabel("Número de tránsito")
plt.ylabel("O - C (minutos)")
plt.title("Transit Timing Variations (TTVs)")
plt.show()

print("Rango TTV (min):", (O_minus_C.max()-O_minus_C.min())*24*60)

In [ ]:
P = best_period_1
t0 = bls.transit_time[np.argmax(bls.power)]

transit_number = np.floor((Times - t0) / P + 0.5).astype(int)
phase = ((Times - t0 + 0.5*P) % P)/P - 0.5
plt.figure(figsize=(8,5))
plt.scatter(phase, Fl_det, s=5, color='k', alpha=0.5)
plt.xlabel("Fase")
plt.ylabel("Flujo")
plt.title(f"Curva plegada P = {P:.4f} d")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))

unique_transits = np.unique(transit_number)

for n in unique_transits:
    mask = transit_number == n
    if np.sum(mask) > 20:  # evitar tránsitos con pocos puntos
        plt.scatter(phase[mask], Fl_det[mask], s=8, alpha=0.7)

plt.xlabel("Fase")
plt.ylabel("Flujo")
plt.title("Tránsitos individuales (colores distintos)")
plt.show()

In [ ]:
P2 = 1.3842282052199513  # tu periodo LS residual

phase2 = (t_out % P2) / P2

plt.figure(figsize=(8,5))
plt.scatter(phase2, f_out, s=5, color='k')
plt.xlabel("Fase")
plt.ylabel("Flujo residual")
plt.title(f"Residuales plegados a P = {P2:.4f} d")
plt.show()


In [ ]:
P1 = best_period_1
t0_1 = bls.transit_time[np.argmax(bls.power)]

phase1 = ((Times - t0_1 + 0.5*P1) % P1)/P1 - 0.5

# puntos en tránsito (ajusta ventana si hace falta)
in_tr1 = np.abs(phase1) < 0.03
out_tr1 = np.abs(phase1) > 0.1

depth1 = np.median(Fl_det[out_tr1]) - np.median(Fl_det[in_tr1])

print("Profundidad tránsito principal:", depth1)


In [ ]:
P2 = 1.3842282052199513  # tu periodo LS de residuales
t0_2 = t_out[np.argmin(f_out)]

phase2 = ((t_out - t0_2 + 0.5*P2) % P2)/P2 - 0.5

in_tr2 = np.abs(phase2) < 0.03
out_tr2 = np.abs(phase2) > 0.1

depth2 = np.median(f_out[out_tr2]) - np.median(f_out[in_tr2])

print("Profundidad segundo evento:", depth2)


In [ ]:
RpRs1 = np.sqrt(depth1)
RpRs2 = np.sqrt(depth2)

print("Rp/Rs tránsito principal:", RpRs1)
print("Rp/Rs segundo evento:", RpRs2)


In [ ]:
best_index = np.argmax(bls.power)
t0_transit = bls.transit_time[best_index]

print("t0 calculado por BLS:", t0_transit)


In [ ]:
P = P_planet

# Número de tránsito para cada punto
n = np.round((t_clean - t_clean.mean()) / P)

# Tiempo calculado de cada tránsito
t_calc = t_clean.mean() + n * P

# Residuales
O_C = t_clean - t_calc

# Nos quedamos solo con puntos dentro del tránsito
in_transit = f_clean < np.percentile(f_clean, 5)

t0_transit = np.mean(t_clean[in_transit] - n[in_transit]*P)

print("t0 refinado:", t0_transit)


In [ ]:
Rs = 0.414543  # por ejemplo, en radios solares

Rp1 = RpRs1 * Rs
Rp2 = RpRs2 * Rs

print("Rp1 en radios solares:", Rp1)
print("Rp2 en radios solares:", Rp2)
